# Practicum 3 - Neural Networks met AmesHousing

In deze notebook bouwen we een neural network om `SalePrice` te voorspellen met de AmesHousing dataset.

Onderzoeksvragen:

- Welke features geven de beste voorspellingen?
- Zijn alle features beter dan een kleinere subset?
- Wat is het verschil met linear regression uit eerdere weken?
- Wat gebeurt er als we learning rate, epochs, validation size en 1/2/3 hidden layers aanpassen?

De code gebruikt dezelfde stijl als de eerdere practica: eerst feature-sets/configuraties definieren, daarna een loop over alle experimenten, en tot slot een samenvattingstabel en grafieken.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_STATE = 42
DATA_FILE = Path("AmesHousing.xlsx")
LOG_FILE = Path("experiment_log.xlsx")

pd.set_option("display.max_columns", 100)

## 1. Data laden

De dataset staat in dezelfde map als deze notebook. We voorspellen `SalePrice`; `ID` gebruiken we niet als feature omdat dit alleen een rij-identificatie is.

In [ ]:
df = pd.read_excel(DATA_FILE)

print("Shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))

In [ ]:
TARGET = "SalePrice"
DROP_COLUMNS = ["ID", TARGET]

numeric_columns = df.drop(columns=DROP_COLUMNS, errors="ignore").select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.drop(columns=DROP_COLUMNS, errors="ignore").select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric_columns)
print("Categorical features:", categorical_columns)

## 2. Feature-sets

Hier definieren we meerdere feature-sets. Dit voorkomt copy-paste: je voegt alleen een extra item toe aan `feature_sets`, en de experiment-loop neemt hem automatisch mee.

In [ ]:
feature_sets = [
    {
        "name": "week6_neighborhood_basic",
        "description": "Lijkt op eerdere linear regression: Lot Area + Year Built + Neighborhood.",
        "features": ["Lot Area", "Year Built", "Neighborhood"],
    },
    {
        "name": "week6_house_style_basic",
        "description": "Lijkt op eerdere linear regression: Year Built + Full Bath + House Style.",
        "features": ["Year Built", "Full Bath", "House Style"],
    },
    {
        "name": "quality_living_basement",
        "description": "Sterke numerieke huiskenmerken: kwaliteit, woonoppervlak en kelderoppervlak.",
        "features": ["Overall Qual", "Gr Liv Area", "Total Bsmt SF"],
    },
    {
        "name": "size_age_rooms",
        "description": "Numerieke subset met grootte, bouwjaar en kamers/badkamers.",
        "features": ["Lot Area", "Gr Liv Area", "Total Bsmt SF", "Year Built", "Full Bath", "Bedroom AbvGr"],
    },
    {
        "name": "all_numeric",
        "description": "Alle numerieke features, zonder ID en SalePrice.",
        "features": numeric_columns,
    },
    {
        "name": "all_features",
        "description": "Alle beschikbare features, behalve ID en SalePrice.",
        "features": numeric_columns + categorical_columns,
    },
]

# Controleer of alle gekozen kolommen bestaan.
for feature_set in feature_sets:
    missing = [feature for feature in feature_set["features"] if feature not in df.columns]
    if missing:
        raise ValueError(f"Feature-set {feature_set['name']} mist kolommen: {missing}")

pd.DataFrame(
    {
        "feature_set": item["name"],
        "description": item["description"],
        "feature_count": len(item["features"]),
        "features": ", ".join(item["features"]),
    }
    for item in feature_sets
)

## 3. Helper-functies

Deze functies houden de experiment-loop klein. De preprocessor schaalt numerieke kolommen en one-hot-encode categorische kolommen. Voor het neural network wordt ook de target (`SalePrice`) geschaald, omdat huizenprijzen grote waarden hebben en een MLP dan stabieler traint.

In [ ]:
def make_one_hot_encoder():
    # Nieuwere sklearn gebruikt sparse_output; oudere versies gebruiken sparse.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X):
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    transformers = []
    if numeric_features:
        transformers.append(("num", StandardScaler(), numeric_features))
    if categorical_features:
        transformers.append(("cat", make_one_hot_encoder(), categorical_features))

    return ColumnTransformer(transformers=transformers, remainder="drop")


def make_train_validation_test_split(X, y, validation_size, test_size=0.15):
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=validation_size,
        random_state=RANDOM_STATE,
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "mse": mse,
        "rmse": np.sqrt(mse),
        "r2": r2_score(y_true, y_pred),
    }


def build_nn_model(X_train, hidden_layers, learning_rate, epochs, batch_size=32, alpha=0.0001):
    preprocessor = build_preprocessor(X_train)
    mlp = MLPRegressor(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        solver="adam",
        learning_rate_init=learning_rate,
        max_iter=epochs,
        batch_size=batch_size,
        alpha=alpha,
        early_stopping=False,
        random_state=RANDOM_STATE,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("nn", mlp),
        ]
    )

    return TransformedTargetRegressor(
        regressor=pipeline,
        transformer=StandardScaler(),
    )

## 4. Baseline: linear regression

Voordat we neural networks beoordelen, trainen we per feature-set ook een LinearRegression baseline. Daardoor kunnen we eerlijk vergelijken met eerdere ML-resultaten.

In [ ]:
baseline_rows = []
baseline_models = {}

for dataset_index, feature_set in enumerate(feature_sets, start=1):
    X = df[feature_set["features"]].copy()
    y = df[TARGET].copy()

    X_train, X_val, X_test, y_train, y_val, y_test = make_train_validation_test_split(
        X,
        y,
        validation_size=0.20,
    )

    baseline_model = Pipeline(
        steps=[
            ("preprocess", build_preprocessor(X_train)),
            ("linear_regression", LinearRegression()),
        ]
    )

    baseline_model.fit(X_train, y_train)
    val_pred = baseline_model.predict(X_val)
    test_pred = baseline_model.predict(X_test)

    val_metrics = regression_metrics(y_val, val_pred)
    test_metrics = regression_metrics(y_test, test_pred)

    baseline_rows.append(
        {
            "dataset_index": dataset_index,
            "feature_set": feature_set["name"],
            "feature_count": len(feature_set["features"]),
            "model_type": "LinearRegression",
            "validation_size": 0.20,
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "val_r2": val_metrics["r2"],
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "test_r2": test_metrics["r2"],
        }
    )

    baseline_models[feature_set["name"]] = baseline_model

baseline_df = pd.DataFrame(baseline_rows).sort_values("test_rmse").reset_index(drop=True)
display(baseline_df)

## 5. Neural-network experimenten

Een `MLPRegressor` is een feed-forward neural network. `hidden_layer_sizes=(64, 32)` betekent 2 hidden layers: eerst 64 neurons, daarna 32 neurons. We testen bewust 1, 2 en 3 hidden layers, verschillende learning rates, epochs en validation sizes.

In [ ]:
nn_settings = [
    {"hidden_layers": (16,), "learning_rate": 0.001, "epochs": 80, "validation_size": 0.15, "batch_size": 32},
    {"hidden_layers": (32,), "learning_rate": 0.001, "epochs": 120, "validation_size": 0.20, "batch_size": 32},
    {"hidden_layers": (64,), "learning_rate": 0.001, "epochs": 160, "validation_size": 0.30, "batch_size": 32},
    {"hidden_layers": (32, 16), "learning_rate": 0.001, "epochs": 120, "validation_size": 0.15, "batch_size": 32},
    {"hidden_layers": (64, 32), "learning_rate": 0.001, "epochs": 160, "validation_size": 0.20, "batch_size": 32},
    {"hidden_layers": (128, 64), "learning_rate": 0.0005, "epochs": 200, "validation_size": 0.20, "batch_size": 32},
    {"hidden_layers": (32, 16), "learning_rate": 0.01, "epochs": 120, "validation_size": 0.20, "batch_size": 32},
    {"hidden_layers": (64, 32), "learning_rate": 0.0001, "epochs": 200, "validation_size": 0.30, "batch_size": 32},
    {"hidden_layers": (64, 32, 16), "learning_rate": 0.001, "epochs": 160, "validation_size": 0.15, "batch_size": 32},
    {"hidden_layers": (128, 64, 32), "learning_rate": 0.0005, "epochs": 200, "validation_size": 0.30, "batch_size": 32},
]

pd.DataFrame(nn_settings)

In [ ]:
results = []
trained_runs = []
experiment_id = 0

for dataset_index, feature_set in enumerate(feature_sets, start=1):
    X = df[feature_set["features"]].copy()
    y = df[TARGET].copy()

    for setting_index, setting in enumerate(nn_settings, start=1):
        experiment_id += 1
        print(
            f"Experiment {experiment_id}: {feature_set['name']} | "
            f"layers={setting['hidden_layers']} | lr={setting['learning_rate']} | "
            f"epochs={setting['epochs']} | val={setting['validation_size']}"
        )

        X_train, X_val, X_test, y_train, y_val, y_test = make_train_validation_test_split(
            X,
            y,
            validation_size=setting["validation_size"],
        )

        model = build_nn_model(
            X_train,
            hidden_layers=setting["hidden_layers"],
            learning_rate=setting["learning_rate"],
            epochs=setting["epochs"],
            batch_size=setting["batch_size"],
        )
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)

        train_metrics = regression_metrics(y_train, train_pred)
        val_metrics = regression_metrics(y_val, val_pred)
        test_metrics = regression_metrics(y_test, test_pred)

        nn_step = model.regressor_.named_steps["nn"]
        final_loss = nn_step.loss_curve_[-1] if hasattr(nn_step, "loss_curve_") else np.nan
        n_iter = nn_step.n_iter_ if hasattr(nn_step, "n_iter_") else np.nan

        row = {
            "experiment_id": experiment_id,
            "dataset_index": dataset_index,
            "setting_index": setting_index,
            "model_type": "MLPRegressor",
            "feature_set": feature_set["name"],
            "feature_count": len(feature_set["features"]),
            "features": ", ".join(feature_set["features"]),
            "hidden_layers": str(setting["hidden_layers"]),
            "hidden_layer_count": len(setting["hidden_layers"]),
            "hidden_nodes_total": sum(setting["hidden_layers"]),
            "learning_rate": setting["learning_rate"],
            "epochs": setting["epochs"],
            "validation_size": setting["validation_size"],
            "batch_size": setting["batch_size"],
            "train_rows": len(X_train),
            "validation_rows": len(X_val),
            "test_rows": len(X_test),
            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "train_r2": train_metrics["r2"],
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "val_r2": val_metrics["r2"],
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
            "test_r2": test_metrics["r2"],
            "final_training_loss": final_loss,
            "actual_iterations": n_iter,
        }
        results.append(row)

        trained_runs.append(
            {
                "row": row,
                "model": model,
                "X_test": X_test,
                "y_test": y_test,
                "test_pred": test_pred,
                "loss_curve": getattr(nn_step, "loss_curve_", []),
            }
        )

results_df = pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)
display(results_df.head(15))

## 6. Resultaten vergelijken

Gebruik de validation set om te kiezen, en controleer daarna de test set als laatste check. Als de testscore veel slechter is dan de validationscore, kan er sprake zijn van overfitting of een ongelukkige split.

In [ ]:
best_by_validation = results_df.iloc[0]
best_by_test = results_df.sort_values("test_rmse").iloc[0]

print("Beste op validation RMSE")
display(best_by_validation.to_frame().T)

print("Beste op test RMSE")
display(best_by_test.to_frame().T)

feature_summary = (
    results_df
    .groupby("feature_set", as_index=False)
    .agg(
        best_val_rmse=("val_rmse", "min"),
        best_test_rmse=("test_rmse", "min"),
        best_test_r2=("test_r2", "max"),
        experiments=("experiment_id", "count"),
    )
    .sort_values("best_val_rmse")
)

display(feature_summary)

In [ ]:
comparison_rows = []

for feature_set_name in feature_summary["feature_set"]:
    baseline_row = baseline_df[baseline_df["feature_set"] == feature_set_name].sort_values("test_rmse").iloc[0]
    nn_row = results_df[results_df["feature_set"] == feature_set_name].sort_values("val_rmse").iloc[0]

    comparison_rows.append(
        {
            "feature_set": feature_set_name,
            "linear_test_rmse": baseline_row["test_rmse"],
            "linear_test_r2": baseline_row["test_r2"],
            "best_nn_validation_rmse": nn_row["val_rmse"],
            "best_nn_test_rmse": nn_row["test_rmse"],
            "best_nn_test_r2": nn_row["test_r2"],
            "rmse_improvement_vs_linear": baseline_row["test_rmse"] - nn_row["test_rmse"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values("best_nn_test_rmse").reset_index(drop=True)
display(comparison_df)

## 7. Visualisaties

De eerste grafiek vergelijkt de beste NN-resultaten per feature-set. Daarna bekijken we de voorspellingen en loss curve van het beste model.

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = feature_summary.sort_values("best_val_rmse")
plt.bar(plot_df["feature_set"], plot_df["best_val_rmse"], color="steelblue", alpha=0.8)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Best validation RMSE")
plt.title("Beste NN validation RMSE per feature-set")
plt.tight_layout()
plt.show()

In [ ]:
best_experiment_id = int(best_by_validation["experiment_id"])
best_run = next(run for run in trained_runs if run["row"]["experiment_id"] == best_experiment_id)

prediction_df = pd.DataFrame(
    {
        "Actual SalePrice": best_run["y_test"].values,
        "Predicted SalePrice": best_run["test_pred"],
    }
)

plt.figure(figsize=(7, 7))
plt.scatter(prediction_df["Actual SalePrice"], prediction_df["Predicted SalePrice"], alpha=0.55)
min_value = min(prediction_df.min())
max_value = max(prediction_df.max())
plt.plot([min_value, max_value], [min_value, max_value], color="red")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title(f"Beste NN experiment {best_experiment_id}: actual vs predicted")
plt.tight_layout()
plt.show()

display(prediction_df.head(10))

In [ ]:
loss_curve = best_run["loss_curve"]

plt.figure(figsize=(8, 4))
plt.plot(loss_curve)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title(f"Loss curve - experiment {best_experiment_id}")
plt.tight_layout()
plt.show()

In [ ]:
comparison_plot = comparison_df.sort_values("feature_set")
x = np.arange(len(comparison_plot))
width = 0.35

plt.figure(figsize=(11, 5))
plt.bar(x - width / 2, comparison_plot["linear_test_rmse"], width, label="Linear regression", alpha=0.8)
plt.bar(x + width / 2, comparison_plot["best_nn_test_rmse"], width, label="Best NN", alpha=0.8)
plt.xticks(x, comparison_plot["feature_set"], rotation=35, ha="right")
plt.ylabel("Test RMSE")
plt.title("Linear regression vs beste neural network per feature-set")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Automatisch Excel-logboek exporteren

Na het draaien van de notebook wordt `experiment_log.xlsx` gevuld met alle experimenten, baselines, best-per-feature en voorspellingen van het beste model.

In [ ]:
with pd.ExcelWriter(LOG_FILE, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="NN experiments", index=False)
    baseline_df.to_excel(writer, sheet_name="Linear baseline", index=False)
    feature_summary.to_excel(writer, sheet_name="Best per feature", index=False)
    comparison_df.to_excel(writer, sheet_name="Linear vs NN", index=False)
    prediction_df.to_excel(writer, sheet_name="Best predictions", index=False)

print(f"Excel-logboek opgeslagen als: {LOG_FILE.resolve()}")

## 9. Conclusie invullen

Vul dit stuk aan nadat je de notebook hebt gedraaid.

Voorbeeldvragen:

- Welke feature-set had de laagste validation RMSE?
- Was `all_features` echt beter dan de subsets?
- Welke learning rate werkte het best?
- Ging een model met 3 hidden layers beter of juist slechter?
- Hoe groot was het verschil tussen linear regression en het beste neural network?